# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nileshkushwaha1410/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup — connect DuckDB to the warehouse release

Lane (from w01/w02): **Refresh / Content Opportunity Scoring**. This contract covers the
warehouse-scale version of that lane — `fact_content_daily_performance`, `dim_content`,
`dim_clients` — developed on the mid-panel partition `month=2026-03`, per the skill's warning
that the `_sample` table is the sealed final month and never safe for label logic.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

%pip -q install duckdb

import os, getpass
import duckdb

# Token order: env var -> Colab Secret -> prompt. Never paste a token into a cell (public repo!).
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
# Mid-panel month, per the assignment's own example -- NOT the _sample table (that's the sealed
# final month, the natural outcome window of any past->future label).
FACT_MARCH  = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

print(con.sql(f"SELECT COUNT(*) AS rows_in_march_partition FROM {FACT_MARCH}").df())


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row (base grain):** one content item's search performance on one day, for one client —
`report_date × client_hash_id × content_hash_id` — from `fact_content_daily_performance`. This is
the DAILY grain I build features from; the thing my lane actually scores is one row per
**content item**, built by aggregating many of these daily rows over a window (same idea as the
starter CSV's `content_id`-level rows, just built by me instead of shipped pre-aggregated).

**Table(s) I'll use:** `fact_content_daily_performance` (the daily fact, base signal) joined to
`dim_content` (content-level joins/context) and `dim_clients` (`gsc_data_start`, `ga4_data_start`,
access flags — needed because this is an unbalanced panel). I am **not** touching
`fact_content_query_90d` this notebook — see "excluded" below.

**Time window:** developing on the mid-panel partition **`month=2026-03`**
(2026-03-01 – 2026-03-31), exactly as the assignment names it. The full daily panel spans
2025-01-27 → 2026-06-30; the final month (2026-06, mirrored by
`fact_content_daily_performance_sample`) is the sealed test month I never touch for label logic.

**What I'd predict/rank (label or proxy):** same proxy shape as my w01/w02 lane framing —
`is_declining_label`: GSC impressions in the more-recent half of the window dropped more than 20%
versus the earlier half, computed strictly from `gsc_impressions` (never from a FlyRank product
decision flag — none ship in this data anyway, per the lane guide). The eventual output is a
ranked opportunity score per content item, not the raw 0/1 label itself.

**One thing I deliberately exclude:** `fact_content_query_90d`, the whole table. Its 90-day window
overlaps the final ~3 months of the snapshot, which overlaps any last-window label I'd build near
the end of the panel — the lane guide flags this exact leakage watch. Rather than carefully split
it into safe/unsafe halves this week, I leave it out of this lane entirely.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Confirm the tables and the EXACT columns I'm relying on above -- don't take the docs on faith.
for name, rel in {
    "fact_content_daily_performance (month=2026-03)": FACT_MARCH,
    "dim_content": DIM_CONTENT,
    "dim_clients": DIM_CLIENTS,
}.items():
    cols = con.sql(f"DESCRIBE SELECT * FROM {rel} LIMIT 0").df()["column_name"].tolist()
    print(name)
    print(" ", cols)
    print()


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature** (knowable before the decision moment — built from the *earlier* half of March only):
- `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` — summed/averaged over days 1–15 (`prev`).
- `active_days_prev` — count of days 1–15 with `gsc_impressions > 0` (a coverage signal, derived
  from the prev window only).
- `ga4_data_available` (context flag, used as a feature) — whether GA4 tracking was live at all
  during the prev window; reflects tracking setup, not an outcome of what I'm predicting.

**Label / proxy** (never a feature):
- `gsc_impressions` summed over days 16–31 (`last`) — the raw ingredient `is_declining_label` is
  computed from. `imp_last` itself, and the label built from it, never enter the feature list.

**Context** (grouping/joining/splitting only — never model inputs):
- `client_hash_id`, `content_hash_id`, `report_date` — used for `GROUP BY`, joins, and would be
  used for client-holdout splits in a real train/test setup.

**Excluded** (one line of why, each):
- `fact_content_query_90d` (whole table) — window overlap with the label period; see Section 1.
- `dim_content.keyword_hash_id`, `dim_content.url_hash_id` — pseudonyms for grouping/dedup only;
  never features, per the data-use terms (no re-identification attempts).
- Rows before a client's `gsc_data_start` / `ga4_data_start` — not truly "zero," just untracked;
  handled by the availability filter in Section 3, never zero-filled and modeled as if observed.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

field_contract = {
    "feature": [
        "gsc_impressions (days 1-15 of March, i.e. prev)",
        "gsc_clicks (prev)",
        "gsc_avg_position (prev)",
        "active_days_prev (coverage, derived from prev only)",
        "ga4_data_available (context flag used as a feature)",
    ],
    "label_or_proxy": [
        "gsc_impressions (days 16-31 of March, i.e. last) -> is_declining_label",
    ],
    "context_only": ["client_hash_id", "content_hash_id", "report_date"],
    "excluded": [
        "fact_content_query_90d (whole table) -- window overlaps the label period",
        "dim_content.keyword_hash_id -- pseudonym, grouping/dedup only",
        "dim_content.url_hash_id -- pseudonym, grouping/dedup only",
        "rows before gsc_data_start/ga4_data_start -- untracked, not zero",
    ],
}
for bucket, fields in field_contract.items():
    print(f"{bucket.upper()}:")
    for f in fields:
        print("  -", f)
    print()


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three checks, each a real query on `month=2026-03`:

1. **Grain** — group by the unit columns and look for duplicates. Zero rows back means the grain
   holds.
2. **Counts + date span** — total rows in the slice, `MIN`/`MAX(report_date)`, and how many
   distinct clients/content items that actually touches.
3. **Availability** — filter `ga4_data_available IS TRUE` (never `= TRUE`, since the flag can be
   `NULL` for content that predates a client's `ga4_data_start`, and `= TRUE` silently drops those
   rows instead of counting them as unavailable) and show how many rows survive.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# --- Query 1: grain probe ---
grain_probe = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT_MARCH}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("QUERY 1 -- grain probe (expect 0 rows back if the grain holds):")
print(grain_probe)
print(f"-> duplicate-grain rows found: {len(grain_probe)}")
print()

# --- Query 2: row count + date span for the slice ---
counts = con.sql(f"""
    SELECT
        COUNT(*)                          AS n_rows,
        MIN(report_date)                  AS min_date,
        MAX(report_date)                  AS max_date,
        COUNT(DISTINCT client_hash_id)    AS n_clients,
        COUNT(DISTINCT content_hash_id)   AS n_content_items
    FROM {FACT_MARCH}
""").df()
print("QUERY 2 -- slice row count + date span (month=2026-03):")
print(counts)
print()

# --- Query 3: availability, filtered IS TRUE (not = TRUE -- the flag can be NULL) ---
availability = con.sql(f"""
    SELECT
        COUNT(*)                                                      AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)   AS ga4_available_rows,
        ROUND(SUM(CASE WHEN ga4_data_available IS TRUE THEN 1.0 ELSE 0 END)
              / COUNT(*), 3)                                          AS ga4_available_share
    FROM {FACT_MARCH}
""").df()
print("QUERY 3 -- availability check (IS TRUE, survivors vs total):")
print(availability)


### 3a. Five features (max), built from that same month

Split March roughly in half — `prev` = days 1–15, `last` = days 16–31 — so `last` gives the trap
below a genuine leak candidate. A minimum-volume floor (`imp_prev >= 50`) keeps the "decline"
reading from being pure noise on low-traffic items, per the lane guide's look-alike checks.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

CUTOFF = "DATE '2026-03-16'"

feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN report_date < {CUTOFF} THEN gsc_impressions ELSE 0 END)      AS imp_prev,
        SUM(CASE WHEN report_date < {CUTOFF} THEN gsc_clicks ELSE 0 END)           AS clk_prev,
        AVG(CASE WHEN report_date < {CUTOFF} THEN gsc_avg_position END)            AS pos_prev,
        COUNT(DISTINCT CASE WHEN report_date < {CUTOFF} AND gsc_impressions > 0
                             THEN report_date END)                                 AS active_days_prev,
        MAX(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)                AS ga4_tracked,
        SUM(CASE WHEN report_date >= {CUTOFF} THEN gsc_impressions ELSE 0 END)     AS imp_last
    FROM {FACT_MARCH}
    GROUP BY client_hash_id, content_hash_id
    HAVING imp_prev >= 50
""").df()

feature_frame["is_declining_label"] = (feature_frame["imp_last"] < 0.8 * feature_frame["imp_prev"]).astype(int)

print(f"Feature frame: {len(feature_frame):,} content items (imp_prev >= 50)")
feature_frame.head()


**Five features, one "available at decision moment because…" line each:**

1. `imp_prev` — total GSC impressions, days 1–15. Available because it's a completed historical
   window, entirely before the `last` window the label is computed from.
2. `clk_prev` — total GSC clicks, days 1–15. Same reasoning: already measured, not future.
3. `pos_prev` — average GSC position, days 1–15. A historical ranking outcome already observed;
   no peeking into days 16–31.
4. `active_days_prev` — count of days 1–15 with impressions > 0. A coverage/consistency signal
   built only from the past window.
5. `ga4_tracked` — whether GA4 tracking was live (`IS TRUE`) at any point in days 1–15. Reflects
   tracking setup, known at decision time — not an outcome of the thing being predicted.

### 3b. The trap — one label-derived column, on purpose

The leakage lesson from notebook 02, performed here on real warehouse data: sneak `imp_last` —
the exact column `is_declining_label` is computed from — into the "feature" list, watch the score
jump toward perfect, then delete it and keep the honest number.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import roc_auc_score

def precision_at_k(scores, y, k):
    order = np.argsort(-scores)[:k]
    y_arr = y.to_numpy() if hasattr(y, "to_numpy") else y
    return y_arr[order].mean()

y = feature_frame["is_declining_label"]

# Honest features only
X_honest = feature_frame[["imp_prev", "clk_prev", "pos_prev", "active_days_prev", "ga4_tracked"]].fillna(0)
honest = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_honest, y)
honest_scores = honest.predict_proba(X_honest)[:, 1]
honest_auc = roc_auc_score(y, honest_scores)
honest_p50 = precision_at_k(honest_scores, y, 50)
print(f"HONEST  -- AUC: {honest_auc:.3f}   Precision@50: {honest_p50:.3f}")
print()

# --- THE TRAP: add imp_last, the label's own raw ingredient, as a "feature" ---
X_leaky = feature_frame[["imp_prev", "clk_prev", "pos_prev", "active_days_prev", "ga4_tracked", "imp_last"]].fillna(0)
leaky = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_leaky, y)
leaky_scores = leaky.predict_proba(X_leaky)[:, 1]
leaky_auc = roc_auc_score(y, leaky_scores)
leaky_p50 = precision_at_k(leaky_scores, y, 50)
print(f"'LEAKY' -- AUC: {leaky_auc:.3f}   Precision@50: {leaky_p50:.3f}   <- suspiciously close to perfect")
print()
print(export_text(leaky, feature_names=list(X_leaky.columns)))
print()
print("The tree just re-derives the label from imp_last, because imp_last IS the label's raw")
print("ingredient: is_declining_label = (imp_last < 0.8 * imp_prev). That's leakage, not signal.")

# Delete the leak. Keep the honest number.
del X_leaky, leaky, leaky_scores, leaky_auc, leaky_p50
print()
print(f"KEPT (honest) -- AUC: {honest_auc:.3f}   Precision@50: {honest_p50:.3f}")


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation: this slice under-represents late-starting clients.** The daily panel is an
unbalanced panel — per-client history depth differs, and `dim_clients.gsc_data_start` /
`ga4_data_start` are the honest per-client start dates. A client whose tracking began after March
2026 contributes zero rows to this month's slice; that isn't "no traffic," it's "not yet tracked."
Any contract or feature I build on `month=2026-03` alone describes only clients already live by
that month, not the full client roster (checked below).

Two more, briefer: the 15/16-day prev/last split here is a dev-speed simplification for a
same-month check — a real capstone label would use the fuller 90-day-style prev30/last30 windows,
which need more than one month of panel. And GA4 non-tracking isn't random: it clusters in
early-history clients, so `ga4_tracked == 0` is a pattern to account for, not noise to fillna away.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

total_clients = con.sql(f"SELECT COUNT(*) FROM {DIM_CLIENTS}").fetchone()[0]
clients_in_march = feature_frame["client_hash_id"].nunique()
ga4_untracked_share = (feature_frame["ga4_tracked"] == 0).mean()

print(f"Total pseudonymized clients in dim_clients:        {total_clients}")
print(f"Clients actually present in this March slice:      {clients_in_march}")
print(f"Share of content items with GA4 NOT tracked prev:   {ga4_untracked_share:.1%}")
print()
print("-> this slice covers only a subset of the full client roster -- the gap is clients whose")
print("   tracking had not started by March 2026, not clients with zero real activity.")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.